# Import Packages

In [ ]:
import pandas as pd
import numpy  as np
import scipy.stats as sp
import matplotlib.pyplot as plt
import pywt

In [ ]:
# Mount your google drive
from google.colab import drive
drive.mount('/content/drive')

## 🔗 Cloning a GitHub Repository

In this notebook, we **clone** the repository into the `/content` folder, which is the working directory in Colab.


In [ ]:
!git clone https://github.com/ljwg3000/UNT_MEEN.git

In [ ]:
data_path = "/content/UNT_MEEN/AI_tutorial/IMS_dataset/2nd_test/"

- **`os` package**: A built-in Python library that allows us to interact with the operating system, such as accessing folders and files.  

- **`os.listdir(path)`**: Returns all file names inside the specified folder.  

- **`sorted(...)`**: Makes sure that the file names are ordered consistently.  

👉 By storing all file names in a list, we can later use a loop (`for`) to load the data files one by one.  
This way, we don’t have to type file names manually, and we also reduce the chance of making mistakes.


In [ ]:
import os

file_list = sorted(os.listdir(data_path))
print("Number of files:", len(file_list))
print(file_list[:3])

## 📊 About the NASA IMS Bearing Dataset (2nd_test)

The dataset we are using comes from the **IMS Bearing Data** collected at the University of Cincinnati’s Center for Intelligent Maintenance Systems (IMS).
It was generated using a test rig where four bearings were mounted on a rotating shaft under constant load and speed until failure occurred.

- **Experiment setup**:  
  - Shaft rotating at 2000 RPM with a radial load of 6000 lbs.  
  - Four Rexnord ZA-2115 bearings installed on the shaft.  
  - Accelerometers mounted on each bearing to record vibration data.  

- **2nd_test dataset**:  
  - Duration: February 12–19, 2004.  
  - **984 files** in total.  
  - **4 channels** (one for each bearing).  
  - Each file contains a **1-second vibration signal** (20,480 points, sampled at 20 kHz).  
  - Data recorded every **10 minutes**.  
  - At the end of the test, **Bearing 1 failed (outer race failure)**.  

👉 In this tutorial, we will use this 2nd test dataset to extract features and visualize the Bearing 1's failure progress.


In [ ]:
for i in range(len(file_list)):
    temp_PATH = data_path + file_list[i]
    temp_DATA = pd.read_csv(temp_PATH, sep='\t', names=['Bearing 1', 'Bearing 2', 'Bearing 3', 'Bearing 4'])
    exec(f"Data_{i+1} = temp_DATA")

Data_1

### Time-domain plots of Bearing 1–4 signals (acceleration)

In [ ]:
time = np.arange(0, 1, 1/20480)

plt.figure(figsize=(12,8))

plt.subplot(4,1,1) #
plt.plot(time , Data_1.iloc[:,0], color='r')
plt.ylabel('Bearing 1', fontsize=12, color='r')
plt.ylim(-1, 1)
plt.grid()

plt.subplot(4,1,2) #
plt.plot(time , Data_1.iloc[:,1], color='g')
plt.ylabel('Bearing 2', fontsize=12, color='g')
plt.ylim(-1, 1)
plt.grid()

plt.subplot(4,1,3) #
plt.plot(time , Data_1.iloc[:,2], color='b')
plt.ylabel('Bearing 3',fontsize=12, color='b')
plt.xlabel('time (s)', fontsize=12)
plt.ylim(-1, 1)
plt.grid()

plt.subplot(4,1,4) #
plt.plot(time , Data_1.iloc[:,3], color='orange')
plt.ylabel('Bearing 4',fontsize=12, color='orange')
plt.xlabel('time (s)', fontsize=12)
plt.ylim(-1, 1)
plt.grid()

plt.show()

### FFT for a data sample

In [ ]:
from scipy.fft import fft, fftfreq

temp_data = Data_100

t = time
x = temp_data.iloc[:,0] # Select a signal (column)

dt = np.mean(np.diff(t))
fs = 1.0 / dt

N = len(x)
X = fft(x)
freq = fftfreq(N, 1/fs)

# single-sided
k_pos = N//2
f_pos = freq[:k_pos]
X_pos = X[:k_pos]

# Amplitude scaling
amp = (2 / N) * np.abs(X_pos)
amp[0] = amp[0] / 2
amp[-1] = amp[-1] / 2

plt.figure(figsize=(12,3))
plt.plot(f_pos, amp, 'b-')
plt.xlim(0, fs/2)
plt.xlabel('Frequency (Hz)', fontsize=12)
plt.ylabel('Amplitude', fontsize=12)
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
from scipy.signal import find_peaks

# suppose we already have f_pos (Hz) and amp (amplitude spectrum)
peaks, props = find_peaks(amp, height=0.005)  # only peaks above 0.005 --> you can adjust the threshold

# peak frequencies & amplitudes
peak_freqs = f_pos[peaks]
peak_amps = amp[peaks]

print("Peak frequencies:", peak_freqs)
print("Peak amplitudes:", peak_amps)

# plot with peaks marked
plt.figure(figsize=(12,3))
plt.plot(f_pos, amp, 'r-')
plt.plot(peak_freqs, peak_amps, 'bo')   # mark peaks
plt.xlim(0, 2000);
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.title("FFT spectrum with detected peaks")
plt.grid(True)
plt.show()

.

.

.

## 🔧 Feature Extraction for Bearing 1

Now we will extract features from the vibration signals of **Bearing 1** only.  
Remember that in the dataset we loaded, there are a total of **984 files (data samples)**, and we will compute time-domain features for each of them.

### **Task for you:**
Use the provided code from **DA3_Code1** as a reference, but **modify it appropriately** for this dataset:
- Use only the **first column** of each data file as the input signal.  
- Exaract 10 features (Max, Min, Mean, RMS, Variance, Skewness, Kurtosis, Crest factor, Shape factor, Impulse factor) for each of the 984 samples.

### **Key points to keep in mind:**
- Unlike the example datasets used in **DA3 and 4**, here the **first column is not time** but directly the signal.  
  - So be careful when applying functions like `np.max(...)` or `rms(...)` — the indexing in `temp_data.iloc[:, j+1]` should be checked carefully..

- We are focusing on **only Bearing 1**, which corresponds to the **first column** of each data file.  
  - This means `NoOfSensor = 1`.  
  - Since there is only one signal, you don’t necessarily need the loop `for j in range(NoOfSensor):`.

- The dataset is **not divided into Normal and Abnormal groups**.  
  - We simply have 984 data samples.  
  - Therefore, there is no need to create separate arrays like `TimeFeature_Normal` or `TimeFeature_Abnormal`. A single `TimeFeature` array will be enough.


By making these adjustments, you will be able to correctly extract features for Bearing 1 from this dataset.


In [ ]:
NoOfData    =   # Number of data samples
NoOfSensor  =   # Only Bearing 1 signal
NoOfFeature =   # 10 types of feature

NoOfData, NoOfSensor, NoOfFeature

In [ ]:
# Define RMS function



In [ ]:
# Create empty(0) arrays for time domin feature dataset (no need to separate normal/abnormal)
# The shape of TimeFeature should be "(10, 984)"

TimeFeature

### **Time Domain Feature Extraction**

In [ ]:


TimeFeature

.

.

### **Frequency Domain Feature Extraction**

- Set MotherWavelet as `sym4`
- Set WT level as `3`
- Again, no need to separate Normal/Abnormal FreqFeature array.
- Again, be careful of indexing when extracting each feature.
   - *There is no time array on the first column.*

In [ ]:
# Wavelet Transform parameter setting
MotherWavelet =    # Mother wavelet
Level         =    # Wavelet decomposition level

In [ ]:
# Create empty(0) arrays for frequency domin feature dataset (no need to separate normal/abnormal)
# The shape of FreqFeature should be "(30, 984)"


FreqFeature

In [ ]:
# Fill FreqFeaure array with features extracted


FreqFeature

In [ ]:
# Complete FeatureData by concatenating TimeFeature and FreqFeature
Features =
FeatureData = pd.DataFrame(Features)
FeatureData.shape

In [ ]:
# Save the FeatureData on your google drive
path = '/content/drive/MyDrive/Colab Notebooks/SavedFiles/FeatureData_Bearing.csv'
FeatureData.to_csv(path, sep=',', header=None , index=None)

.

.

.


## 🔎 Data Visualization with t-SNE

Now that we have extracted time-domain features, we want to **visualize the data distribution** in a lower-dimensional space. To do this, let's apply **t-SNE (t-distributed Stochastic Neighbor Embedding)** to our `FeatureData` you extracted above, referring to **DA4_Code2**.

### Steps to follow:

1. **Import necessary packages**  
   - We need `StandardScaler` from `sklearn.preprocessing` to normalize the feature data.  
   - We need `TSNE` from `sklearn.manifold` to perform the dimensionality reduction.  

2. **Normalize the feature data**  
   - Before applying t-SNE, it is important to normalize all features so they have the same scale.  
   - ⚠️ **Important:** Make sure the data matrix is structured as  
     - **rows = samples** (here, 984 data samples)  
     - **columns = features** (the extracted feature values).  
   - If your `FeatureData` is not in this format, you may need to **transpose** it.

3. **Apply t-SNE**  
   - Set the number of components (`n_components=2`) so that the features are reduced into a 2D space.  
   - Set the `perplexity` as `20` and `number of iterations` as `700` to control the quality of the embedding.

4. **Visualize with scatter plot**  
   - Create a 2D scatter plot.  
   - Each point corresponds to one data sample.


In [ ]:
# Standardize the FeatureData


FeatureData_std =
pd.DataFrame(FeatureData_std)

In [ ]:
# Implement the t-SNE


tsne_results =


In [ ]:
# Visualize Bearing 1 data
plt.figure(figsize=(8,6))



plt.show()

.

.

.

<details>
<summary>
Adding a Colorbar to Show Trends
</summary>

When we plot the t-SNE results, each point corresponds to one data sample.  
To show how the samples progress over time, we can **map the sample index to a color**.

- In `plt.scatter(...)`, the argument `c=order` assigns a different color to each sample based on its order (early → late).  
- The argument `cmap='RdYlGn_r'` specifies the color scale (here, green → red).  

By adding a **colorbar**, we can clearly see the trend:

```python
N = tsne_results.shape[0]                 # 984
order = np.arange(N)                      # 0,1,2,...,983 (early -> late)

plt.figure(figsize=(8,6))
sc = plt.scatter(tsne_results[:,0], tsne_results[:,1],
                 c=order, cmap='RdYlGn_r',  # reversed so low=green, high=red
                 s=18, alpha=0.85)

plt.title('t-SNE Result', fontsize=15)
plt.xlabel('t-SNE_1')
plt.ylabel('t-SNE_2')
plt.grid(alpha=0.5)

cb = plt.colorbar(sc, pad=0.02)
cb.set_label('Sample index (early → late)')
plt.show()


<details>
<summary>
Interactive Visualization Tool
</summary>

`plt.scatter` works well, but it produces a **static image** — you cannot zoom in, hover over points, or see extra details.



### 🔎 Plotly Package

**Plotly Express** is a library that makes plots **interactive**:

- You can **hover your mouse** over each point to see its index and coordinates.  
- You can **zoom in/out** and **pan** around the scatter plot.  
- The color of each point is mapped to its **sample index**, so you can visually track how the data changes over time.  
- A **color bar** is automatically added, which helps interpret the meaning of the colors.  



### ⚖️ Matplotlib vs. Plotly (simple comparison)

- **Matplotlib** → static plots, good for quick and simple visualizations.  
- **Plotly** → interactive plots, better for exploring data in more detail.  

---
Try to copy/paste and run the code below.

```python

import plotly.express as px

N = tsne_results.shape[0]
df_plot = pd.DataFrame({
    "x": tsne_results[:,0],
    "y": tsne_results[:,1],
    "index": np.arange(N)   # Sample index (0→N-1)
})

fig = px.scatter(
    df_plot, x="x", y="y", color="index",
    color_continuous_scale="RdYlGn_r",
    hover_data={"index": True, "x":":.2f", "y":":.2f"},
    labels={"index":"Sample index", "x":"t-SNE_1", "y":"t-SNE_2"},
)

fig.update_traces(marker=dict(size=7, opacity=0.85))
fig.update_layout(title="t-SNE Result", font=dict(size=15))
fig.show()